# Legal Contract Intelligence Pipeline — Walkthrough

This notebook demonstrates the end-to-end pipeline for extracting key clauses and generating structured summaries from legal contracts in the [CUAD dataset](https://www.atticusprojectai.org/cuad).

## What this notebook covers
1. Environment setup and imports
2. Loading a single contract from CUAD
3. Text preprocessing walkthrough
4. Clause extraction with Gemini (JSON mode)
5. Contract summarization
6. Running the full 50-contract pipeline
7. Evaluating against CUAD ground truth (Token F1)
8. Semantic search over extracted clauses

**Prerequisites:** Complete setup in README.md (virtual env, `pip install -r requirements.txt`, `.env` with `GEMINI_API_KEY`).

## 1. Setup

In [ ]:
import os
import json
import textwrap
from dotenv import load_dotenv

load_dotenv()  # loads GEMINI_API_KEY from .env

# Verify API key is set
api_key = os.getenv('GEMINI_API_KEY', '')
if not api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY not found.\n"
        "Copy .env.example → .env and add your key.\n"
        "Free key: https://ai.google.dev"
    )
print(f"API key configured: {'*' * (len(api_key) - 4)}{api_key[-4:]}")

## 2. Load a Single Contract from CUAD

In [ ]:
from loader import load_from_huggingface

# Load 1 contract for demonstration
# First run auto-downloads CUAD from HuggingFace (~200 MB)
print("Loading CUAD dataset (first run downloads ~200 MB)...")
contracts = load_from_huggingface(n_contracts=1)
contract = contracts[0]

print(f"\nContract ID: {contract['id']}")
print(f"Source:      {contract['source']}")
print(f"Text length: {len(contract['text']):,} characters")
print(f"\nFirst 500 chars:")
print("-" * 60)
print(contract['text'][:500])

## 3. Text Preprocessing

In [ ]:
from preprocessor import normalize_text, truncate_for_context

raw_text = contract['text']
clean_text = normalize_text(raw_text)
clean_text = truncate_for_context(clean_text)

print(f"Raw    : {len(raw_text):,} chars")
print(f"Clean  : {len(clean_text):,} chars")
print(f"Saved  : {len(raw_text) - len(clean_text):,} chars removed (whitespace/control chars)")

In [ ]:
# Demo: hyphenation repair
import re, unicodedata

broken   = "This Agreement may be termi-\nnated upon sixty days notice."
repaired = re.sub(r"-\s*\n\s*([a-z])", r"\1", broken)
print(f"Before: {broken!r}")
print(f"After:  {repaired!r}")

## 4. Clause Extraction

In [ ]:
import google.generativeai as genai
from config import GEMINI_MODEL, EXTRACT_TEMPERATURE
from extractor import extract_clauses

genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model = genai.GenerativeModel(GEMINI_MODEL)

print(f"Model: {GEMINI_MODEL}  |  Temperature: {EXTRACT_TEMPERATURE} (deterministic)")
print("Extracting clauses...")

clauses = extract_clauses(model, clean_text)

print("\n" + "=" * 60)
for clause_type, text in clauses.items():
    label = clause_type.replace('_', ' ').title()
    print(f"\n{label}:")
    print("-" * 40)
    if text in {"NOT_FOUND", "EXTRACTION_FAILED"}:
        print(f"  [{text}]")
    else:
        print(textwrap.fill(text[:500], width=80, initial_indent="  "))
        if len(text) > 500:
            print(f"  ... [{len(text)-500} more chars]")

## 5. Contract Summarization

In [ ]:
from summarizer import summarize_contract

print("Generating summary...")
summary = summarize_contract(model, clean_text)

word_count = len(summary.split())
print(f"\nSummary ({word_count} words):")
print("-" * 60)
print(textwrap.fill(summary, width=80))

## 6. Full Pipeline — 5 Contracts

In [ ]:
# Process 5 contracts (change n=50 for full run)
# Full 50-contract run takes ~8-10 minutes (API rate limiting)
from pipeline import run_pipeline

results = run_pipeline(
    n_contracts=5,
    output_dir='output_demo',
    enable_search=True
)

print(f"\nProcessed {len(results)} contracts")
print(f"Output files: output_demo/results.csv, output_demo/results.json")

In [ ]:
# Preview results
import pandas as pd

df = pd.DataFrame(results)
display(df[['contract_id', 'summary']].head())
print(f"\nShape: {df.shape}")
print("\nSentinel value counts:")
for col in ['termination_clause', 'confidentiality_clause', 'liability_clause']:
    not_found  = (df[col] == 'NOT_FOUND').sum()
    failed     = (df[col] == 'EXTRACTION_FAILED').sum()
    extracted  = len(df) - not_found - failed
    print(f"  {col}: {extracted} extracted, {not_found} NOT_FOUND, {failed} FAILED")

## 7. Evaluation — Token F1 vs CUAD Ground Truth

In [ ]:
from evaluator import evaluate_results, print_report
from loader import load_from_huggingface

# Load same contracts with ground-truth answers
contracts_with_gt = load_from_huggingface(n_contracts=5)

report = evaluate_results(results, contracts_with_gt)
print_report(report)

## 8. Semantic Search (BONUS)

In [ ]:
from search import ClauseSearchEngine

engine = ClauseSearchEngine()
n_indexed = engine.index(results)
print(f"Indexed {n_indexed} clauses")

In [ ]:
# Search all clause types
query = "termination without cause or notice period"
hits = engine.search(query, top_k=3)

print(f'Query: "{query}"\n')
for hit in hits:
    print(f"  Score: {hit['similarity_score']:.4f}")
    print(f"  Contract: {hit['contract_id']}")
    print(f"  Type: {hit['clause_type']}")
    print(f"  Text: {hit['clause_text'][:200]}...")
    print()

In [ ]:
# Search with clause type filter
query = "indemnification for intellectual property infringement"
hits = engine.search(query, top_k=3, clause_type="liability_clause")

print(f'Query: "{query}"  [filter: liability_clause]\n')
for hit in hits:
    print(f"  [{hit['similarity_score']:.4f}] {hit['contract_id']}")
    print(f"  {hit['clause_text'][:300]}...\n")

## Summary

| Component | Implementation | Key Feature |
|---|---|---|
| Data loading | `loader.py` | CUAD HuggingFace + PyMuPDF PDF |
| Preprocessing | `preprocessor.py` | NFKC + hyphenation repair |
| Clause extraction | `extractor.py` + `prompts.py` | JSON mode, 2-shot, CoT, safe fill() |
| Summarization | `summarizer.py` | 100-150 words, 3 structured sections |
| Evaluation | `evaluator.py` | Token F1 (CUAD standard metric) |
| Semantic search | `search.py` | MiniLM + ChromaDB cosine similarity |

Run the full pipeline:
```bash
python main.py --n 50
```

Evaluate results:
```bash
python evaluator.py --results output/results.json
```